# Vis-Head Causality: Qwen2.5-VL, Gemma-4, InternVL3.5 (<=8B)

Compares head-level causal steering effects across three VLM families, all
checkpoints <=8B parameters, two sizes per family:

- **Qwen2.5-VL**: 3B-Instruct, 7B-Instruct
- **Gemma-4**: E2B-it, E4B-it
- **InternVL3.5**: 4B, 8B

## Grid alignment (Task 1)

Every family's vision encoder was probed directly (not assumed from published
config) to find a canvas size where the merged visual-token grid divides
evenly into G=4 non-overlapping cells for all three families simultaneously:
**448x448px canvas -> 16x16=256 merged tokens for all three families**, so
G=4 (2x2 grid, 8x8=64 tokens/cell) needs **no gutter** anywhere -- confirmed
by direct probing (see `/tmp/probe_task1_step1*.py`,
`/tmp/probe_canvas_sizes.py`, `/tmp/probe_task1_step4_rowmajor*.py` for the
full probe scripts and results this was derived from). Row-major token
ordering was empirically verified (not assumed) via a solid-color-block
localization test: Qwen2.5-VL and InternVL3.5 both show dozens of heads with
*perfect* 4/4 cross-placement tracking; Gemma-4's mapping is derived directly
from `Gemma4VisionPooler`'s real pooling kernel-index math (not an
assumption), but the empirical attention signal for it was noisier (best
head 3/4, 0/336 perfect) -- **flagged as a caveat, not disproven**, and
accepted per explicit confirmation.

## Scoring (Task 2)

Five per-head quantities are computed and compared: raw target mass (the
original score), target share (mean-of-ratios and pooled-ratio variants),
total visual attention, and **excess mass** `S(R*) - (1/G)*sum_R S(R)` (the
new primary score -- zero for uniform/no-attention heads, large only when a
head both attends to the image and concentrates on the target). Top-K
overlap between raw-mass and excess-mass selections is reported per
checkpoint.

## Layer-matched control (Task 3)

Causal effects are compared against `R_CTRL` control head sets that match
the VIR selection's per-layer histogram exactly (same count of heads per
layer, drawn from that layer's non-selected heads), rather than a uniformly-
random baseline that would confound "causally special" with "happens to sit
deeper." Controls are drawn once and applied identically across all
evaluation samples. Permutation p-value floor = `1/(R_CTRL+1)`, never quoted
below that floor.

## Splits

Head selection runs on discovery samples (`rng` seeded `SEED+100`); causal
evaluation runs on a disjoint set of MCQ samples (`rng` seeded `SEED+555`) --
different random draws by construction, effectively zero collision
probability given the ImageNet-grid sampling space.

In [1]:
%matplotlib inline
import gc
import re
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis_head").exists():
    raise RuntimeError("Run this notebook from the repository root (vis-head/).")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from scipy import stats as sstats
from tqdm.auto import tqdm

from vis_head.common import DEFAULT_SEED
from vis_head.vir import (
    aggregate_region_attention, collect_last_query_attentions, rank_heads_by_score,
    per_sample_head_scores, pooled_target_share,
)
from vis_head.imagenet_grid import DEFAULT_IMAGENET_ROOT, list_val_class_dirs, load_class_names, sample_grid
from vis_head.modeling import decode_generated_text, find_image_token_range, load_model_and_processor, model_dims, prepare_inputs, run_generation
from vis_head.regions import assign_grid_cells_to_tokens, region_positions_from_ids
from vis_head.steering import (
    group_heads_by_layer, intervention_positions, make_static_attention_mask_hook, register_mask_hooks, remove_handles,
    layer_matched_random_controls, assert_layer_histogram_matches, permutation_pvalue_floor,
)

DEVICE = "cuda:0"
SEED = DEFAULT_SEED

# ---- Task 1: grid alignment ----
# 448x448px canvas -> 16x16=256 merged tokens for Qwen2.5-VL, Gemma-4, AND
# InternVL3.5 simultaneously (directly probed, not assumed -- see notebook
# intro). G=4 (2x2 grid) divides 16 exactly: cell_size=224px, gap=0px, no
# gutter needed for any family.
CANVAS = 448
ROWS, COLS = 2, 2
N_CELLS = ROWS * COLS   # G=4
CELL_SIZE = CANVAS // ROWS   # 224px, exact, gap=0
GRID_GAP = 0

N_OPTIONS = 4
OPTION_LETTERS = ["A", "B", "C", "D"][:N_OPTIONS]

# ---- Task 2: K as a fraction of total heads, not a fixed count ----
K_FRACTION = 0.10   # 10% of L*H heads selected per checkpoint

# ---- Task 3: layer-matched control ----
R_CTRL = 5     # FAST, non-final run per explicit request -- not the optimal N.

DISCOVERY_PROMPT = lambda name: f"What shows the {name}?"   # single fixed discovery phrasing, validated best (18-phrasing sweep)

imagenet_class_dirs = list_val_class_dirs(DEFAULT_IMAGENET_ROOT)
imagenet_class_names = load_class_names(DEFAULT_IMAGENET_ROOT)
print(f"{len(imagenet_class_dirs)} ImageNet classes available")


def build_mcq_sample(rng):
    grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, gap=GRID_GAP, rng=rng,
                        class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    assert grid.grid.size == (CANVAS, CANVAS), f"Grid canvas {grid.grid.size} != expected ({CANVAS},{CANVAS})"
    target_cell = int(rng.randint(N_CELLS))
    correct_name = grid.cell_names[target_cell]
    other_names = [n for i, n in enumerate(grid.cell_names) if i != target_cell]
    n_distractors = min(N_OPTIONS - 1, len(other_names))
    distractor_idx = rng.choice(len(other_names), size=n_distractors, replace=False)
    distractors = [other_names[i] for i in distractor_idx]
    options = [correct_name] + distractors
    order = rng.permutation(len(options))
    options = [options[i] for i in order]
    correct_letter = OPTION_LETTERS[int(np.where(order == 0)[0][0])]
    option_lines = "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS[:len(options)], options))
    prompt = f"Which of the following is shown in this image? {option_lines}. Answer with only the letter."
    return {"grid": grid, "target_cell": target_cell, "options": options, "correct_letter": correct_letter, "prompt": prompt}


def extract_letter_last(text, valid_letters):
    matches = re.findall(r"\b([" + "".join(valid_letters) + r"])\b", text.upper())
    return matches[-1] if matches else None


def macro_f1(y_true, y_pred):
    labels = sorted(set(y_true) | set(y_pred))
    return float(f1_score(y_true, y_pred, labels=labels, average="macro"))


def macro_auc(y_true, probs, classes):
    y_true_bin = label_binarize(y_true, classes=classes)
    try:
        return float(roc_auc_score(y_true_bin, probs, average="macro", multi_class="ovr"))
    except ValueError:
        return float("nan")


def score_stats(scores):
    flat = scores.reshape(-1)
    return {"mean": float(flat.mean()), "median": float(np.median(flat)), "std": float(flat.std()), "max": float(flat.max())}


def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()


def get_letter_token_ids(tokenizer, letters):
    ids = {}
    for letter in letters:
        candidates = set()
        for cand in (letter, f" {letter}"):
            enc = tokenizer.encode(cand, add_special_tokens=False)
            if len(enc) == 1:
                candidates.add(enc[0])
        if not candidates:
            raise ValueError(f"No single-token encoding found for letter {letter!r}")
        ids[letter] = sorted(candidates)
    return ids


def run_mcq_generate(model, inputs, prompt_length, heads_by_layer, register_hooks_fn,
                      max_new_tokens, letter_token_ids, all_letter_ids_flat, id_to_letter):
    handles = register_hooks_fn(heads_by_layer) if heads_by_layer is not None else []
    try:
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                  output_scores=True, return_dict_in_generate=True)
    finally:
        remove_handles(handles)

    gen_ids = out.sequences[0, prompt_length:].tolist()
    answer_step = None
    for step in range(len(gen_ids) - 1, -1, -1):
        if gen_ids[step] in all_letter_ids_flat:
            answer_step = step
            break
    if answer_step is None:
        return {"predicted": None, "probs": np.zeros(N_OPTIONS), "correct": False}

    predicted = id_to_letter[gen_ids[answer_step]]
    step_logits = out.scores[answer_step][0].float()
    letter_logits = [step_logits[letter_token_ids[l]].max().item() for l in OPTION_LETTERS]
    probs = torch.softmax(torch.tensor(letter_logits), dim=0).numpy()
    return {"predicted": predicted, "probs": probs, "correct": None}


def mcnemar_p(cond_a_correct, cond_b_correct):
    b = sum(1 for a, c in zip(cond_a_correct, cond_b_correct) if not a and c)
    c = sum(1 for a, c in zip(cond_a_correct, cond_b_correct) if a and not c)
    n = b + c
    if n == 0:
        return float("nan")
    stat = (abs(b - c) - 1) ** 2 / n
    return float(sstats.chi2.sf(stat, df=1))


all_results = {}   # (family, checkpoint_tag) -> summary dict

1000 ImageNet classes available


/mnt/abka03/.conda/internvl_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
# Section 1: Qwen2.5-VL (3B-Instruct, 7B-Instruct)

Native dynamic-resolution vision tokens; patch grid geometry read directly
from `image_grid_thw` + `spatial_merge_size` at runtime (never assumed).
On the 448x448 canvas this notebook uses, both sizes produce an exact 16x16
merged token grid (verified in Task 1 probing).

In [2]:
# ----------------------------- Section 1 configuration -----------------------------
QWEN25VL_CHECKPOINTS = {
    "qwen25vl_3b": {"model_id": "Qwen/Qwen2.5-VL-3B-Instruct", "max_new_tokens": 6},
    "qwen25vl_7b": {"model_id": "Qwen/Qwen2.5-VL-7B-Instruct", "max_new_tokens": 6},
}
QWEN25VL_N_DISCOVERY = 300
QWEN25VL_N_CAUSAL = 300


def run_qwen25vl_checkpoint(tag, cfg, n_discovery, n_causal):
    model_id_str = cfg["model_id"]
    print(f"\n=== [{tag}] Loading {model_id_str} ===")
    model, processor = load_model_and_processor(model_id=model_id_str, device=DEVICE)
    n_layers, n_heads, spatial_merge = model_dims(model)
    total_heads = n_layers * n_heads
    K = max(1, round(K_FRACTION * total_heads))
    print(f"{n_layers} layers x {n_heads} heads = {total_heads} total  ->  K={K} ({K_FRACTION:.0%})")

    letter_token_ids = get_letter_token_ids(processor.tokenizer, OPTION_LETTERS)
    all_letter_ids_flat = set(i for ids in letter_token_ids.values() for i in ids)
    id_to_letter = {i: l for l, ids in letter_token_ids.items() for i in ids}

    # ---------------- Task 1: grid-aligned discovery (selection split) ----------------
    rng = np.random.RandomState(SEED + 100)
    raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    target_share_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    total_visual_attn_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    excess_mass_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid = 0
    for _ in tqdm(range(n_discovery), desc=f"[{tag}] discovery", leave=False):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, gap=GRID_GAP, rng=rng,
                            class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        assert grid.grid.size == (CANVAS, CANVAS)
        target_cell = int(rng.randint(N_CELLS))
        prompt = DISCOVERY_PROMPT(grid.cell_names[target_cell])
        try:
            inputs = prepare_inputs(processor, grid.grid, prompt, DEVICE)
            t, h, w = inputs["image_grid_thw"][0].tolist()
            g_h, g_w = h // spatial_merge, w // spatial_merge
            # Task 1 acceptance assertion: regions equal-sized, disjoint,
            # exhaustive -- a failed assertion halts the run (no silent
            # per-sample discarding, which would introduce a selection effect).
            img_start, img_end = find_image_token_range(inputs, processor)
            n_image_tokens = img_end - img_start
            assert n_image_tokens == g_h * g_w, (
                f"Image token count {n_image_tokens} != merged grid g_h*g_w={g_h*g_w} "
                f"(grid=({g_h},{g_w})) -- some tokens are unaccounted for."
            )
            assert g_h % 2 == 0 and g_w % 2 == 0, f"Merged grid ({g_h},{g_w}) not evenly divisible by sqrt(G)=2."

            region_ids, _ = assign_grid_cells_to_tokens(image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
            attn = collect_last_query_attentions(model, inputs)
            region_attn = aggregate_region_attention(attn_at_query=attn, inputs=inputs, processor=processor, region_ids=region_ids, n_regions=N_CELLS)
            # Confirm the attention row sums to 1 (the correct tensor/slice is being read).
            row_sum_check = attn[:, :, :].sum(axis=-1)
            assert np.allclose(row_sum_check, 1.0, atol=1e-2), "Attention row does not sum to 1 -- wrong tensor/slice."

            scores = per_sample_head_scores(region_attn, target_cell)
            raw_sum += scores["raw_target_mass"]
            target_share_sum += scores["target_share"]
            total_visual_attn_sum += scores["total_visual_attention"]
            excess_mass_sum += scores["excess_mass"]
            valid += 1
        except AssertionError:
            raise   # Task 1: a failed acceptance assertion must halt the run
        except Exception as exc:
            print(f"  Skipping grid: {exc}")
    print(f"  [{tag}] discovery valid={valid}/{n_discovery}")

    raw_scores = (raw_sum / max(valid, 1)).astype(np.float32)
    mean_target_share = (target_share_sum / max(valid, 1)).astype(np.float32)
    pooled_share = pooled_target_share(raw_sum, total_visual_attn_sum).astype(np.float32)
    total_visual_attn = (total_visual_attn_sum / max(valid, 1)).astype(np.float32)
    excess_mass = (excess_mass_sum / max(valid, 1)).astype(np.float32)

    # ---------------- Task 2: raw-mass vs excess-mass top-K overlap ----------------
    ranked_raw = rank_heads_by_score(raw_scores)
    ranked_excess = rank_heads_by_score(excess_mass)
    top_raw = set((r["layer"], r["head"]) for r in ranked_raw[:K])
    top_excess = set((r["layer"], r["head"]) for r in ranked_excess[:K])
    overlap_frac = len(top_raw & top_excess) / K
    print(f"  [{tag}] top-K overlap (raw-mass vs excess-mass): {len(top_raw & top_excess)}/{K} = {overlap_frac:.2f}")

    vir_heads = [(r["layer"], r["head"]) for r in ranked_excess[:K]]   # excess-mass is primary per Task 2
    heads_by_layer = group_heads_by_layer(vir_heads)
    rel_depths = sorted(set(round(l / n_layers, 3) for l, h in vir_heads))
    print(f"  [{tag}] VIR head relative depths (l/L): {rel_depths}")

    # ---------------- Task 3: layer-matched control (drawn once) ----------------
    ctrl_rng = np.random.RandomState(SEED + 777)
    control_draws = layer_matched_random_controls(vir_heads, n_layers, n_heads, R_CTRL, ctrl_rng)
    for draw in control_draws:
        assert_layer_histogram_matches(vir_heads, draw)
    print(f"  [{tag}] {R_CTRL} layer-matched control draws, all histograms verified")

    # ---------------- causal evaluation (disjoint evaluation split) ----------------
    mcq_rng = np.random.RandomState(SEED + 555)
    mcq_samples = [build_mcq_sample(mcq_rng) for _ in range(n_causal)]

    def run_one(sample, heads_by_layer_arg):
        inputs = prepare_inputs(processor, sample["grid"].grid, sample["prompt"], DEVICE)
        prompt_length = int(inputs["input_ids"].shape[1])

        def hooks_fn(hbl):
            img_start, img_end = find_image_token_range(inputs, processor)
            region_ids, _ = assign_grid_cells_to_tokens(image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
            positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
            target_positions = positions[sample["target_cell"]]
            other_positions = [p for i in range(N_CELLS) if i != sample["target_cell"] for p in positions[i]]
            suppress_positions, boost_positions, pad = intervention_positions(
                mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
                img_start=img_start, img_end=img_end, prompt_length=prompt_length)
            hook_by_layer = {
                l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                    boost_positions=boost_positions, n_query_heads=n_heads,
                                                    device=DEVICE, decode_only=False, pad_with_suppress=pad)
                for l, hh in hbl.items()
            }
            return register_mask_hooks(model, hook_by_layer)

        result = run_mcq_generate(model, inputs, prompt_length, heads_by_layer_arg, hooks_fn,
                                   cfg["max_new_tokens"], letter_token_ids, all_letter_ids_flat, id_to_letter)
        result["correct"] = result["predicted"] == sample["correct_letter"]
        return result

    baseline_res, vir_res = [], []
    for sample in tqdm(mcq_samples, desc=f"[{tag}] MCQ baseline+VIR", leave=False):
        baseline_res.append(run_one(sample, None))
        vir_res.append(run_one(sample, heads_by_layer))

    # layer-matched controls: applied identically across all samples, drawn once above
    control_accs, control_correct_lists = [], []
    for draw in tqdm(control_draws, desc=f"[{tag}] layer-matched controls", leave=False):
        ctrl_by_layer = group_heads_by_layer(draw)
        ctrl_res = [run_one(sample, ctrl_by_layer) for sample in mcq_samples]
        ctrl_correct = [r["correct"] for r in ctrl_res]
        control_correct_lists.append(ctrl_correct)
        control_accs.append(float(np.mean(ctrl_correct)))

    y_true = [s["correct_letter"] for s in mcq_samples]
    b_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in baseline_res]
    v_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in vir_res]
    b_probs = np.stack([r["probs"] for r in baseline_res])
    v_probs = np.stack([r["probs"] for r in vir_res])
    b_correct = [r["correct"] for r in baseline_res]
    v_correct = [r["correct"] for r in vir_res]

    baseline_acc = float(np.mean(b_correct))
    vir_acc = float(np.mean(v_correct))
    control_acc_mean = float(np.mean(control_accs))
    control_acc_std = float(np.std(control_accs))
    # permutation p-value: fraction of control draws whose accuracy >= VIR's,
    # floored at 1/(R_CTRL+1), matching the standard permutation-test convention.
    n_ge = sum(1 for a in control_accs if a >= vir_acc)
    perm_p = max((n_ge + 1) / (R_CTRL + 1), permutation_pvalue_floor(R_CTRL))
    p_vir_vs_baseline = mcnemar_p(b_correct, v_correct)

    stats = score_stats(raw_scores)
    summary = {
        "family": "qwen25vl", "checkpoint": tag, "model_id": model_id_str,
        "n_layers": n_layers, "n_heads": n_heads, "K": K,
        "topk_overlap_raw_vs_excess": overlap_frac,
        "vh_raw_mean": stats["mean"], "vh_raw_max": stats["max"],
        "baseline_acc": baseline_acc, "vir_acc": vir_acc,
        "control_acc_mean": control_acc_mean, "control_acc_std": control_acc_std,
        "p_vir_vs_baseline_mcnemar": p_vir_vs_baseline,
        "p_vir_vs_layer_matched_control": perm_p,
        "baseline_f1": macro_f1(y_true, b_pred), "vir_f1": macro_f1(y_true, v_pred),
        "baseline_auc": macro_auc(y_true, b_probs, OPTION_LETTERS), "vir_auc": macro_auc(y_true, v_probs, OPTION_LETTERS),
    }
    print(f"  [{tag}] baseline_acc={baseline_acc:.3f}  VIR_acc={vir_acc:.3f}  "
          f"control_acc={control_acc_mean:.3f}+-{control_acc_std:.3f}  "
          f"p(VIR vs control)={perm_p:.2e}  p(VIR vs baseline)={p_vir_vs_baseline:.2e}")

    free_gpu(model, processor)
    return summary

## Section 1 -- Pilot run (small N, catches bugs fast)

In [3]:
PILOT_N_DISCOVERY = 20
PILOT_N_CAUSAL = 20
PILOT_R_CTRL_SAVE = R_CTRL
R_CTRL = 10   # small for pilot speed; restored to full value before main run

for tag, cfg in QWEN25VL_CHECKPOINTS.items():
    result = run_qwen25vl_checkpoint(tag, cfg, PILOT_N_DISCOVERY, PILOT_N_CAUSAL)
    all_results[("qwen25vl", tag + "_PILOT")] = result

R_CTRL = PILOT_R_CTRL_SAVE
pd.DataFrame([v for k, v in all_results.items() if k[0] == "qwen25vl" and "PILOT" in k[1]])


=== [qwen25vl_3b] Loading Qwen/Qwen2.5-VL-3B-Instruct ===


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

36 layers x 16 heads = 576 total  ->  K=29 (5%)


...

  [qwen25vl_7b] discovery valid=20/20
  [qwen25vl_7b] top-K overlap (raw-mass vs excess-mass): 24/39 = 0.62
  [qwen25vl_7b] VIR head relative depths (l/L): [0.5, 0.536, 0.571, 0.643, 0.679, 0.714, 0.75, 0.786, 0.857, 0.929, 0.964]
  [qwen25vl_7b] 10 layer-matched control draws, all histograms verified


[qwen25vl_7b] MCQ baseline+VIR:   0%|          | 0/20 [00:00<?, ?it/s]

[qwen25vl_7b] layer-matched controls:   0%|          | 0/10 [00:00<?, ?it/s]

  [qwen25vl_7b] baseline_acc=0.300  VIR_acc=0.150  control_acc=0.355+-0.133  p(VIR vs control)=9.09e-01  p(VIR vs baseline)=3.71e-01


,family,checkpoint,model_id,n_layers,n_heads,K,topk_overlap_raw_vs_excess,vh_raw_mean,vh_raw_max,baseline_acc,vir_acc,control_acc_mean,control_acc_std,p_vir_vs_baseline_mcnemar,p_vir_vs_layer_matched_control,baseline_f1,vir_f1,baseline_auc,vir_auc
0,qwen25vl,qwen25vl_3b,Qwen/Qwen2.5-VL-3B-Instruct,36,16,29,0.275862,0.027747,0.223303,0.2,0.45,0.245,0.131244,0.130570,0.272727,0.184343,0.411905,0.578691,0.685020
1,qwen25vl,qwen25vl_7b,Qwen/Qwen2.5-VL-7B-Instruct,28,28,39,0.615385,0.022414,0.263712,0.3,0.15,0.355,0.133135,0.371093,0.909091,0.230263,0.155345,0.532460,0.409953


## Section 1 -- Main run (N=QWEN25VL_N_DISCOVERY discovery, QWEN25VL_N_CAUSAL causal, default 600/600)

In [3]:
for tag, cfg in QWEN25VL_CHECKPOINTS.items():
    try:
        result = run_qwen25vl_checkpoint(tag, cfg, QWEN25VL_N_DISCOVERY, QWEN25VL_N_CAUSAL)
        all_results[("qwen25vl", tag)] = result
    except ValueError as e:
        print(f"[{tag}] SKIPPED -- layer-matched control infeasible at this K_FRACTION: {e}")
        free_gpu()
        continue

qwen25vl_df = pd.DataFrame([v for k, v in all_results.items() if k[0] == "qwen25vl" and "PILOT" not in k[1]])
qwen25vl_df.to_csv("logs/multistage_qwen25vl_results.csv", index=False)
print(qwen25vl_df.to_string(index=False))


=== [qwen25vl_3b] Loading Qwen/Qwen2.5-VL-3B-Instruct ===


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

36 layers x 16 heads = 576 total  ->  K=29 (5%)


...

[qwen25vl_7b] discovery:   0%|          | 0/300 [00:00<?, ?it/s]

  [qwen25vl_7b] discovery valid=300/300
  [qwen25vl_7b] top-K overlap (raw-mass vs excess-mass): 22/39 = 0.56
  [qwen25vl_7b] VIR head relative depths (l/L): [0.5, 0.571, 0.607, 0.643, 0.679, 0.714, 0.75, 0.786, 0.857, 0.929, 0.964]
  [qwen25vl_7b] 5 layer-matched control draws, all histograms verified


[qwen25vl_7b] MCQ baseline+VIR:   0%|          | 0/300 [00:00<?, ?it/s]

[qwen25vl_7b] layer-matched controls:   0%|          | 0/5 [00:00<?, ?it/s]

  [qwen25vl_7b] baseline_acc=0.243  VIR_acc=0.310  control_acc=0.359+-0.136  p(VIR vs control)=6.67e-01  p(VIR vs baseline)=2.72e-02
  family  checkpoint                    model_id  n_layers  n_heads  K  topk_overlap_raw_vs_excess  vh_raw_mean  vh_raw_max  baseline_acc  vir_acc  control_acc_mean  control_acc_std  p_vir_vs_baseline_mcnemar  p_vir_vs_layer_matched_control  baseline_f1   vir_f1  baseline_auc  vir_auc
qwen25vl qwen25vl_3b Qwen/Qwen2.5-VL-3B-Instruct        36       16 29                    0.275862     0.027856    0.208813      0.256667 0.346667          0.222667         0.038839                   0.001054                        0.166667     0.257900 0.331434      0.487798 0.588191
qwen25vl qwen25vl_7b Qwen/Qwen2.5-VL-7B-Instruct        28       28 39                    0.564103     0.022389    0.299370      0.243333 0.310000          0.359333         0.135571                   0.027195                        0.666667     0.201819 0.309505      0.479467 0.560306


---
# Section 2: Gemma-4 (E2B-it, E4B-it)

Aspect-ratio-adaptive vision pooling -- the processor does not expose the
post-pool 2D grid shape directly, only pre-pool `image_position_ids` and the
final soft-token count. `gemma4_output_grid` (in `vis_head.demo_adapters`,
already validated in the interactive-steering demo) re-derives the post-pool
`(g_h, g_w)` from the same kernel-index math `Gemma4VisionPooler` itself
uses. On this notebook's 448x448 canvas, this reliably lands on 16x16=256
tokens (verified across a 12-point canvas-size sweep, Task 1).

**Caveat (per explicit confirmation, kept as a standing note):** row-major
token ordering for Gemma-4 is derived directly from the real pooling kernel
math (not assumed), but the empirical solid-color-block localization test
found no head with perfect 4/4 cross-placement tracking (best: 3/4, 0/336
perfect) -- weaker than Qwen2.5-VL and InternVL3.5's clean confirmations
(dozens of perfect heads each). Treated as correct by construction; flagged
here as an open empirical uncertainty specific to Gemma-4.

In [2]:
# ----------------------------- Section 2 configuration -----------------------------
from transformers import AutoModelForImageTextToText, AutoProcessor as _AutoProcessor2
import vis_head.vir as _vir_module
import vis_head.modeling as _modeling_module
from vis_head.demo_adapters import gemma4_output_grid

GEMMA4_CHECKPOINTS = {
    "gemma4_e2b_it": {"model_id": "google/gemma-4-E2B-it", "max_new_tokens": 6},
    "gemma4_e4b_it": {"model_id": "google/gemma-4-E4B-it", "max_new_tokens": 6},
}
GEMMA4_N_DISCOVERY = 300
GEMMA4_N_CAUSAL = 300


def gemma4_prepare_inputs(processor, image, prompt, device):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    return processor.apply_chat_template([messages], tokenize=True, add_generation_prompt=True,
                                          return_tensors="pt", return_dict=True).to(device)


def gemma4_assign_grid(g_h, g_w):
    fake_thw = torch.tensor([[1, g_h, g_w]])
    return assign_grid_cells_to_tokens(image_grid_thw=fake_thw, rows=ROWS, cols=COLS, spatial_merge=1)


def run_gemma4_checkpoint(tag, cfg, n_discovery, n_causal):
    model_id_str = cfg["model_id"]
    print(f"\n=== [{tag}] Loading {model_id_str} ===")
    processor = _AutoProcessor2.from_pretrained(model_id_str)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id_str, torch_dtype=torch.bfloat16, attn_implementation="eager"
    ).to(DEVICE)
    model.eval()
    n_layers = len(model.model.language_model.layers)
    n_heads = model.config.text_config.num_attention_heads
    total_heads = n_layers * n_heads
    K = max(1, round(K_FRACTION * total_heads))
    print(f"{n_layers} layers x {n_heads} heads = {total_heads} total  ->  K={K} ({K_FRACTION:.0%})")

    image_token_id = model.config.image_token_id

    def gemma4_find_image_token_range(inputs, processor=None):
        ids = inputs["input_ids"][0].tolist()
        positions = [i for i, t in enumerate(ids) if t == image_token_id]
        if not positions:
            raise ValueError("No Gemma-4 image tokens found.")
        return positions[0], positions[-1] + 1

    # dual monkey-patch: aggregate_region_attention uses the module-level
    # import in vis_head.vir; other call sites may use the lazy import from
    # vis_head.modeling -- both need patching for full Gemma-4 compatibility.
    _vir_module.find_image_token_range = gemma4_find_image_token_range
    _modeling_module.find_image_token_range = gemma4_find_image_token_range

    letter_token_ids = get_letter_token_ids(processor.tokenizer, OPTION_LETTERS)
    all_letter_ids_flat = set(i for ids in letter_token_ids.values() for i in ids)
    id_to_letter = {i: l for l, ids in letter_token_ids.items() for i in ids}

    # ---------------- Task 1: grid-aligned discovery (selection split) ----------------
    rng = np.random.RandomState(SEED + 100)
    raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    target_share_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    total_visual_attn_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    excess_mass_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid = 0
    for _ in tqdm(range(n_discovery), desc=f"[{tag}] discovery", leave=False):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, gap=GRID_GAP, rng=rng,
                            class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        assert grid.grid.size == (CANVAS, CANVAS)
        target_cell = int(rng.randint(N_CELLS))
        prompt = DISCOVERY_PROMPT(grid.cell_names[target_cell])
        try:
            inputs = gemma4_prepare_inputs(processor, grid.grid, prompt, DEVICE)
            img_start, img_end = gemma4_find_image_token_range(inputs)
            n_image_tokens = img_end - img_start
            g_h, g_w = gemma4_output_grid(inputs["image_position_ids"], n_image_tokens)
            assert n_image_tokens == g_h * g_w, (
                f"Image token count {n_image_tokens} != derived grid g_h*g_w={g_h*g_w} (grid=({g_h},{g_w}))."
            )
            assert g_h % 2 == 0 and g_w % 2 == 0, f"Merged grid ({g_h},{g_w}) not evenly divisible by sqrt(G)=2."

            region_ids, _ = gemma4_assign_grid(g_h, g_w)
            attn = collect_last_query_attentions(model, inputs)
            region_attn = aggregate_region_attention(attn_at_query=attn, inputs=inputs, processor=processor, region_ids=region_ids, n_regions=N_CELLS)
            row_sum_check = attn[:, :, :].sum(axis=-1)
            assert np.allclose(row_sum_check, 1.0, atol=1e-2), "Attention row does not sum to 1 -- wrong tensor/slice."

            scores = per_sample_head_scores(region_attn, target_cell)
            raw_sum += scores["raw_target_mass"]
            target_share_sum += scores["target_share"]
            total_visual_attn_sum += scores["total_visual_attention"]
            excess_mass_sum += scores["excess_mass"]
            valid += 1
        except AssertionError:
            raise
        except Exception as exc:
            print(f"  Skipping grid: {exc}")
    print(f"  [{tag}] discovery valid={valid}/{n_discovery}")

    raw_scores = (raw_sum / max(valid, 1)).astype(np.float32)
    total_visual_attn_sum_f = total_visual_attn_sum
    excess_mass = (excess_mass_sum / max(valid, 1)).astype(np.float32)

    # ---------------- Task 2: raw-mass vs excess-mass top-K overlap ----------------
    ranked_raw = rank_heads_by_score(raw_scores)
    ranked_excess = rank_heads_by_score(excess_mass)
    top_raw = set((r["layer"], r["head"]) for r in ranked_raw[:K])
    top_excess = set((r["layer"], r["head"]) for r in ranked_excess[:K])
    overlap_frac = len(top_raw & top_excess) / K
    print(f"  [{tag}] top-K overlap (raw-mass vs excess-mass): {len(top_raw & top_excess)}/{K} = {overlap_frac:.2f}")

    vir_heads = [(r["layer"], r["head"]) for r in ranked_excess[:K]]
    heads_by_layer = group_heads_by_layer(vir_heads)
    rel_depths = sorted(set(round(l / n_layers, 3) for l, h in vir_heads))
    print(f"  [{tag}] VIR head relative depths (l/L): {rel_depths}")

    # ---------------- Task 3: layer-matched control (drawn once) ----------------
    ctrl_rng = np.random.RandomState(SEED + 777)
    control_draws = layer_matched_random_controls(vir_heads, n_layers, n_heads, R_CTRL, ctrl_rng)
    for draw in control_draws:
        assert_layer_histogram_matches(vir_heads, draw)
    print(f"  [{tag}] {R_CTRL} layer-matched control draws, all histograms verified")

    # ---------------- causal evaluation (disjoint evaluation split) ----------------
    mcq_rng = np.random.RandomState(SEED + 555)
    mcq_samples = [build_mcq_sample(mcq_rng) for _ in range(n_causal)]

    def run_one(sample, heads_by_layer_arg):
        inputs = gemma4_prepare_inputs(processor, sample["grid"].grid, sample["prompt"], DEVICE)
        prompt_length = int(inputs["input_ids"].shape[1])

        def hooks_fn(hbl):
            img_start, img_end = gemma4_find_image_token_range(inputs)
            n_image_tokens = img_end - img_start
            g_h, g_w = gemma4_output_grid(inputs["image_position_ids"], n_image_tokens)
            region_ids, _ = gemma4_assign_grid(g_h, g_w)
            positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
            target_positions = positions[sample["target_cell"]]
            other_positions = [p for i in range(N_CELLS) if i != sample["target_cell"] for p in positions[i]]
            suppress_positions, boost_positions, pad = intervention_positions(
                mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
                img_start=img_start, img_end=img_end, prompt_length=prompt_length)
            hook_by_layer = {
                l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                    boost_positions=boost_positions, n_query_heads=n_heads,
                                                    device=DEVICE, decode_only=False, pad_with_suppress=pad)
                for l, hh in hbl.items()
            }
            return register_mask_hooks(model, hook_by_layer)

        result = run_mcq_generate(model, inputs, prompt_length, heads_by_layer_arg, hooks_fn,
                                   cfg["max_new_tokens"], letter_token_ids, all_letter_ids_flat, id_to_letter)
        result["correct"] = result["predicted"] == sample["correct_letter"]
        return result

    baseline_res, vir_res = [], []
    for sample in tqdm(mcq_samples, desc=f"[{tag}] MCQ baseline+VIR", leave=False):
        baseline_res.append(run_one(sample, None))
        vir_res.append(run_one(sample, heads_by_layer))

    control_accs = []
    for draw in tqdm(control_draws, desc=f"[{tag}] layer-matched controls", leave=False):
        ctrl_by_layer = group_heads_by_layer(draw)
        ctrl_res = [run_one(sample, ctrl_by_layer) for sample in mcq_samples]
        control_accs.append(float(np.mean([r["correct"] for r in ctrl_res])))

    y_true = [s["correct_letter"] for s in mcq_samples]
    b_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in baseline_res]
    v_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in vir_res]
    b_probs = np.stack([r["probs"] for r in baseline_res])
    v_probs = np.stack([r["probs"] for r in vir_res])
    b_correct = [r["correct"] for r in baseline_res]
    v_correct = [r["correct"] for r in vir_res]

    baseline_acc = float(np.mean(b_correct))
    vir_acc = float(np.mean(v_correct))
    control_acc_mean = float(np.mean(control_accs))
    control_acc_std = float(np.std(control_accs))
    n_ge = sum(1 for a in control_accs if a >= vir_acc)
    perm_p = max((n_ge + 1) / (R_CTRL + 1), permutation_pvalue_floor(R_CTRL))
    p_vir_vs_baseline = mcnemar_p(b_correct, v_correct)

    stats = score_stats(raw_scores)
    summary = {
        "family": "gemma4", "checkpoint": tag, "model_id": model_id_str,
        "n_layers": n_layers, "n_heads": n_heads, "K": K,
        "topk_overlap_raw_vs_excess": overlap_frac,
        "vh_raw_mean": stats["mean"], "vh_raw_max": stats["max"],
        "baseline_acc": baseline_acc, "vir_acc": vir_acc,
        "control_acc_mean": control_acc_mean, "control_acc_std": control_acc_std,
        "p_vir_vs_baseline_mcnemar": p_vir_vs_baseline,
        "p_vir_vs_layer_matched_control": perm_p,
        "baseline_f1": macro_f1(y_true, b_pred), "vir_f1": macro_f1(y_true, v_pred),
        "baseline_auc": macro_auc(y_true, b_probs, OPTION_LETTERS), "vir_auc": macro_auc(y_true, v_probs, OPTION_LETTERS),
    }
    print(f"  [{tag}] baseline_acc={baseline_acc:.3f}  VIR_acc={vir_acc:.3f}  "
          f"control_acc={control_acc_mean:.3f}+-{control_acc_std:.3f}  "
          f"p(VIR vs control)={perm_p:.2e}  p(VIR vs baseline)={p_vir_vs_baseline:.2e}")

    free_gpu(model, processor)
    return summary

## Section 2 -- Pilot run (small N, catches bugs fast)

In [3]:
PILOT_N_DISCOVERY = 20
PILOT_N_CAUSAL = 20
PILOT_R_CTRL_SAVE = R_CTRL
R_CTRL = 10

for tag, cfg in GEMMA4_CHECKPOINTS.items():
    result = run_gemma4_checkpoint(tag, cfg, PILOT_N_DISCOVERY, PILOT_N_CAUSAL)
    all_results[("gemma4", tag + "_PILOT")] = result

R_CTRL = PILOT_R_CTRL_SAVE
pd.DataFrame([v for k, v in all_results.items() if k[0] == "gemma4" and "PILOT" in k[1]])


=== [gemma4_e2b_it] Loading google/gemma-4-E2B-it ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

35 layers x 8 heads = 280 total  ->  K=14 (5%)


[gemma4_e2b_it] discovery:   0%|          | 0/20 [00:00<?, ?it/s]

...

  [gemma4_e4b_it] discovery valid=20/20
  [gemma4_e4b_it] top-K overlap (raw-mass vs excess-mass): 9/17 = 0.53
  [gemma4_e4b_it] VIR head relative depths (l/L): [0.595, 0.619, 0.643, 0.69, 0.738, 0.762, 0.81, 0.857, 0.929, 0.952]
  [gemma4_e4b_it] 10 layer-matched control draws, all histograms verified


[gemma4_e4b_it] MCQ baseline+VIR:   0%|          | 0/20 [00:00<?, ?it/s]

[gemma4_e4b_it] layer-matched controls:   0%|          | 0/10 [00:00<?, ?it/s]

  [gemma4_e4b_it] baseline_acc=0.250  VIR_acc=0.650  control_acc=0.275+-0.162  p(VIR vs control)=9.09e-02  p(VIR vs baseline)=2.69e-02


,family,checkpoint,model_id,n_layers,n_heads,K,topk_overlap_raw_vs_excess,vh_raw_mean,vh_raw_max,baseline_acc,vir_acc,control_acc_mean,control_acc_std,p_vir_vs_baseline_mcnemar,p_vir_vs_layer_matched_control,baseline_f1,vir_f1,baseline_auc,vir_auc
0,gemma4,gemma4_e2b_it,google/gemma-4-E2B-it,35,8,14,0.642857,0.051932,0.273637,0.25,0.75,0.375,0.230489,0.004427,0.181818,0.244444,0.693910,0.484125,0.930994
1,gemma4,gemma4_e4b_it,google/gemma-4-E4B-it,42,8,17,0.529412,0.057646,0.253274,0.25,0.65,0.275,0.161632,0.026857,0.090909,0.255556,0.530303,0.530714,0.883966


## Section 2 -- Main run (N=GEMMA4_N_DISCOVERY discovery, GEMMA4_N_CAUSAL causal, default 600/600)

In [3]:
for tag, cfg in GEMMA4_CHECKPOINTS.items():
    try:
        result = run_gemma4_checkpoint(tag, cfg, GEMMA4_N_DISCOVERY, GEMMA4_N_CAUSAL)
        all_results[("gemma4", tag)] = result
    except ValueError as e:
        print(f"[{tag}] SKIPPED -- layer-matched control infeasible at this K_FRACTION: {e}")
        free_gpu()
        continue

gemma4_df = pd.DataFrame([v for k, v in all_results.items() if k[0] == "gemma4" and "PILOT" not in k[1]])
gemma4_df.to_csv("logs/multistage_gemma4_results.csv", index=False)
print(gemma4_df.to_string(index=False))


=== [gemma4_e2b_it] Loading google/gemma-4-E2B-it ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

35 layers x 8 heads = 280 total  ->  K=14 (5%)


[gemma4_e2b_it] discovery:   0%|          | 0/300 [00:00<?, ?it/s]

...

[gemma4_e4b_it] discovery:   0%|          | 0/300 [00:00<?, ?it/s]

  [gemma4_e4b_it] discovery valid=300/300
  [gemma4_e4b_it] top-K overlap (raw-mass vs excess-mass): 10/17 = 0.59
  [gemma4_e4b_it] VIR head relative depths (l/L): [0.595, 0.619, 0.643, 0.69, 0.738, 0.762, 0.786, 0.81, 0.905, 0.952]
  [gemma4_e4b_it] 5 layer-matched control draws, all histograms verified


[gemma4_e4b_it] MCQ baseline+VIR:   0%|          | 0/300 [00:00<?, ?it/s]

[gemma4_e4b_it] layer-matched controls:   0%|          | 0/5 [00:00<?, ?it/s]

  [gemma4_e4b_it] baseline_acc=0.230  VIR_acc=0.400  control_acc=0.271+-0.160  p(VIR vs control)=3.33e-01  p(VIR vs baseline)=9.13e-06
family    checkpoint              model_id  n_layers  n_heads  K  topk_overlap_raw_vs_excess  vh_raw_mean  vh_raw_max  baseline_acc  vir_acc  control_acc_mean  control_acc_std  p_vir_vs_baseline_mcnemar  p_vir_vs_layer_matched_control  baseline_f1   vir_f1  baseline_auc  vir_auc
gemma4 gemma4_e2b_it google/gemma-4-E2B-it        35        8 14                    0.428571     0.043886    0.234929      0.236667 0.253333          0.353333         0.113940                   0.706703                        0.833333     0.229996 0.175005      0.482678 0.474997
gemma4 gemma4_e4b_it google/gemma-4-E4B-it        42        8 17                    0.588235     0.050892    0.230405      0.230000 0.400000          0.271333         0.160424                   0.000009                        0.333333     0.230215 0.352804      0.487952 0.693684


---
# Section 3: InternVL3.5 (4B, 8B)

**Runs in a separate conda environment (`internvl_env`, transformers==4.57.2)**
-- InternVL's custom remote-code `modeling_internvl_chat.py` is incompatible
with the `transformers==5.15.0` used for Qwen2.5-VL/Gemma-4 (Sections 1-2).
This section's cells execute against a `python3` Jupyter kernel registered
under `internvl_env`, not the default kernel these Section 1-2 cells run
under -- see `run_section3_pilot.py` / `run_section3_main.py`.

Single-tile preprocessing: InternVL3.5's own vision config
(`vision_config.image_size=448`, `patch_size=14`, `downsample_ratio=0.5`)
was verified directly against a real forward pass (`extract_feature()`
output shape), not assumed -- and happens to exactly match this notebook's
448x448 canvas, so the composite grid is always processed as InternVL's
native single tile (no dynamic multi-tile splitting triggered). Row-major
token ordering confirmed empirically: 55/1152 heads show perfect 4/4
solid-color-block placement tracking (Task 1).

In [2]:
# ----------------------------- Section 3 configuration -----------------------------
from transformers import AutoModel, AutoTokenizer
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

INTERNVL_CHECKPOINTS = {
    "internvl35_4b": {"model_id": "OpenGVLab/InternVL3_5-4B"},
    "internvl35_8b": {"model_id": "OpenGVLab/InternVL3_5-8B"},
}
INTERNVL_N_DISCOVERY = 300
INTERNVL_N_CAUSAL = 300
INTERNVL_MAX_NEW_TOKENS = 6
INTERNVL_IMAGENET_MEAN = (0.485, 0.456, 0.406)
INTERNVL_IMAGENET_STD = (0.229, 0.224, 0.225)

import vis_head.vir as _vir_module


def internvl_prepare_inputs(model, tokenizer, image, prompt, device, tokens_per_tile, image_size):
    transform = T.Compose([
        T.Resize((image_size, image_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=INTERNVL_IMAGENET_MEAN, std=INTERNVL_IMAGENET_STD),
    ])
    pixel_values = transform(image.convert("RGB")).unsqueeze(0).to(device=device, dtype=model.dtype)
    img_context_token_id = tokenizer.convert_tokens_to_ids("<IMG_CONTEXT>")
    image_tokens = "<IMG_CONTEXT>" * tokens_per_tile
    full_prompt = f"<img>{image_tokens}</img>\n{prompt}"
    system_message = getattr(model, "system_message", "")
    chat_text = f"<|im_start|>system\n{system_message}<|im_end|>\n<|im_start|>user\n{full_prompt}<|im_end|>\n<|im_start|>assistant\n"
    model_inputs = tokenizer(chat_text, return_tensors="pt").to(device)
    return {"input_ids": model_inputs["input_ids"], "attention_mask": model_inputs["attention_mask"],
            "pixel_values": pixel_values, "img_context_token_id": img_context_token_id}


def internvl_find_image_token_range(inputs, processor=None):
    ids = inputs["input_ids"][0].tolist()
    ctx_id = inputs["img_context_token_id"]
    positions = [i for i, t in enumerate(ids) if t == ctx_id]
    if not positions:
        raise ValueError("No InternVL IMG_CONTEXT tokens found.")
    return positions[0], positions[-1] + 1


def internvl_assign_grid(grid_side):
    fake_thw = torch.tensor([[1, grid_side, grid_side]])
    return assign_grid_cells_to_tokens(image_grid_thw=fake_thw, rows=ROWS, cols=COLS, spatial_merge=1)


class _InternVLForwardAdapter:
    '''model.forward(pixel_values=..., input_ids=..., ...) requires image_flags
    to mark which rows of pixel_values are real images -- always 1 tile here,
    so image_flags is a constant ones vector. Wraps the custom-code model
    call so the rest of the pipeline sees a plain model(**inputs) /
    model.generate(**inputs) interface like every other family.'''
    def __init__(self, model):
        self.model = model

    def __call__(self, **kwargs):
        kwargs = dict(kwargs)
        img_context_token_id = kwargs.pop("img_context_token_id", None)
        if img_context_token_id is not None:
            self.model.img_context_token_id = img_context_token_id
        kwargs["image_flags"] = torch.ones(kwargs["pixel_values"].shape[0], dtype=torch.long, device=kwargs["pixel_values"].device)
        return self.model(**kwargs)

    def generate(self, **kwargs):
        kwargs = dict(kwargs)
        img_context_token_id = kwargs.pop("img_context_token_id")
        self.model.img_context_token_id = img_context_token_id
        pixel_values = kwargs.pop("pixel_values")
        input_ids = kwargs.pop("input_ids")
        attention_mask = kwargs.pop("attention_mask")
        return self.model.generate(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, **kwargs)


def run_internvl_checkpoint(tag, cfg, n_discovery, n_causal):
    model_id_str = cfg["model_id"]
    print(f"\n=== [{tag}] Loading {model_id_str} ===")
    tokenizer = AutoTokenizer.from_pretrained(model_id_str, trust_remote_code=True, use_fast=False)
    raw_model = AutoModel.from_pretrained(
        model_id_str, torch_dtype=torch.bfloat16, trust_remote_code=True, attn_implementation="eager"
    ).to(DEVICE)
    raw_model.eval()
    model = _InternVLForwardAdapter(raw_model)
    n_layers = raw_model.config.llm_config.num_hidden_layers
    n_heads = raw_model.config.llm_config.num_attention_heads
    total_heads = n_layers * n_heads
    K = max(1, round(K_FRACTION * total_heads))
    print(f"{n_layers} layers x {n_heads} heads = {total_heads} total  ->  K={K} ({K_FRACTION:.0%})")

    # Task 1: read patch_size/image_size/downsample from the REAL config, and
    # tokens_per_tile from a REAL forward pass -- never hardcoded/assumed.
    real_image_size = raw_model.config.vision_config.image_size
    probe_transform = T.Compose([
        T.Resize((real_image_size, real_image_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(), T.Normalize(mean=INTERNVL_IMAGENET_MEAN, std=INTERNVL_IMAGENET_STD),
    ])
    from PIL import Image as _PILImage
    probe_img = _PILImage.new("RGB", (real_image_size, real_image_size), (128, 128, 128))
    with torch.no_grad():
        probe_pixel_values = probe_transform(probe_img).unsqueeze(0).to(device=DEVICE, dtype=raw_model.dtype)
        tokens_per_tile = raw_model.extract_feature(probe_pixel_values).shape[1]
    grid_side = int(round(tokens_per_tile ** 0.5))
    assert grid_side * grid_side == tokens_per_tile, f"tokens_per_tile={tokens_per_tile} is not a perfect square."
    assert real_image_size == CANVAS, (
        f"InternVL's real image_size={real_image_size} != notebook canvas={CANVAS} -- "
        f"the two were confirmed to match during Task 1 probing; if this fires, that has changed."
    )
    assert grid_side % 2 == 0, f"grid_side={grid_side} not evenly divisible by sqrt(G)=2."
    print(f"  [{tag}] real image_size={real_image_size}  tokens_per_tile={tokens_per_tile}  grid_side={grid_side}")

    _vir_module.find_image_token_range = internvl_find_image_token_range

    letter_token_ids = get_letter_token_ids(tokenizer, OPTION_LETTERS)
    all_letter_ids_flat = set(i for ids in letter_token_ids.values() for i in ids)
    id_to_letter = {i: l for l, ids in letter_token_ids.items() for i in ids}

    # ---------------- Task 1: grid-aligned discovery (selection split) ----------------
    rng = np.random.RandomState(SEED + 100)
    raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    target_share_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    total_visual_attn_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    excess_mass_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid = 0
    for _ in tqdm(range(n_discovery), desc=f"[{tag}] discovery", leave=False):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, gap=GRID_GAP, rng=rng,
                            class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        assert grid.grid.size == (CANVAS, CANVAS)
        target_cell = int(rng.randint(N_CELLS))
        prompt = DISCOVERY_PROMPT(grid.cell_names[target_cell])
        try:
            inputs = internvl_prepare_inputs(raw_model, tokenizer, grid.grid, prompt, DEVICE, tokens_per_tile, real_image_size)
            img_start, img_end = internvl_find_image_token_range(inputs)
            n_image_tokens = img_end - img_start
            assert n_image_tokens == tokens_per_tile, f"matched tokens {n_image_tokens} != tokens_per_tile {tokens_per_tile}"

            region_ids, _ = internvl_assign_grid(grid_side)
            attn = collect_last_query_attentions(model, inputs)
            region_attn = aggregate_region_attention(attn_at_query=attn, inputs=inputs, processor=None, region_ids=region_ids, n_regions=N_CELLS)
            row_sum_check = attn[:, :, :].sum(axis=-1)
            assert np.allclose(row_sum_check, 1.0, atol=1e-2), "Attention row does not sum to 1 -- wrong tensor/slice."

            scores = per_sample_head_scores(region_attn, target_cell)
            raw_sum += scores["raw_target_mass"]
            target_share_sum += scores["target_share"]
            total_visual_attn_sum += scores["total_visual_attention"]
            excess_mass_sum += scores["excess_mass"]
            valid += 1
        except AssertionError:
            raise
        except Exception as exc:
            print(f"  Skipping grid: {exc}")
    print(f"  [{tag}] discovery valid={valid}/{n_discovery}")

    raw_scores = (raw_sum / max(valid, 1)).astype(np.float32)
    excess_mass = (excess_mass_sum / max(valid, 1)).astype(np.float32)

    # ---------------- Task 2: raw-mass vs excess-mass top-K overlap ----------------
    ranked_raw = rank_heads_by_score(raw_scores)
    ranked_excess = rank_heads_by_score(excess_mass)
    top_raw = set((r["layer"], r["head"]) for r in ranked_raw[:K])
    top_excess = set((r["layer"], r["head"]) for r in ranked_excess[:K])
    overlap_frac = len(top_raw & top_excess) / K
    print(f"  [{tag}] top-K overlap (raw-mass vs excess-mass): {len(top_raw & top_excess)}/{K} = {overlap_frac:.2f}")

    vir_heads = [(r["layer"], r["head"]) for r in ranked_excess[:K]]
    heads_by_layer = group_heads_by_layer(vir_heads)
    rel_depths = sorted(set(round(l / n_layers, 3) for l, h in vir_heads))
    print(f"  [{tag}] VIR head relative depths (l/L): {rel_depths}")

    # ---------------- Task 3: layer-matched control (drawn once) ----------------
    ctrl_rng = np.random.RandomState(SEED + 777)
    control_draws = layer_matched_random_controls(vir_heads, n_layers, n_heads, R_CTRL, ctrl_rng)
    for draw in control_draws:
        assert_layer_histogram_matches(vir_heads, draw)
    print(f"  [{tag}] {R_CTRL} layer-matched control draws, all histograms verified")

    # ---------------- causal evaluation (disjoint evaluation split) ----------------
    mcq_rng = np.random.RandomState(SEED + 555)
    mcq_samples = [build_mcq_sample(mcq_rng) for _ in range(n_causal)]

    def run_one(sample, heads_by_layer_arg):
        inputs = internvl_prepare_inputs(raw_model, tokenizer, sample["grid"].grid, sample["prompt"], DEVICE, tokens_per_tile, real_image_size)

        def hooks_fn(hbl):
            img_start, img_end = internvl_find_image_token_range(inputs)
            region_ids, _ = internvl_assign_grid(grid_side)
            positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
            target_positions = positions[sample["target_cell"]]
            other_positions = [p for i in range(N_CELLS) if i != sample["target_cell"] for p in positions[i]]
            suppress_positions, boost_positions, pad = intervention_positions(
                mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
                img_start=img_start, img_end=img_end, prompt_length=0)
            hook_by_layer = {
                l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                    boost_positions=boost_positions, n_query_heads=n_heads,
                                                    device=DEVICE, decode_only=False, pad_with_suppress=pad)
                for l, hh in hbl.items()
            }
            return register_mask_hooks(raw_model, hook_by_layer)

        # InternVL's custom generate() returns ONLY newly-generated tokens
        # (sequences.shape == (1, max_new_tokens), unlike every other family
        # in this notebook) -- confirmed via direct inspection. prompt_length=0
        # here is therefore correct, not a bug.
        result = run_mcq_generate(model, inputs, 0, heads_by_layer_arg, hooks_fn,
                                   INTERNVL_MAX_NEW_TOKENS, letter_token_ids, all_letter_ids_flat, id_to_letter)
        result["correct"] = result["predicted"] == sample["correct_letter"]
        return result

    baseline_res, vir_res = [], []
    for sample in tqdm(mcq_samples, desc=f"[{tag}] MCQ baseline+VIR", leave=False):
        baseline_res.append(run_one(sample, None))
        vir_res.append(run_one(sample, heads_by_layer))

    control_accs = []
    for draw in tqdm(control_draws, desc=f"[{tag}] layer-matched controls", leave=False):
        ctrl_by_layer = group_heads_by_layer(draw)
        ctrl_res = [run_one(sample, ctrl_by_layer) for sample in mcq_samples]
        control_accs.append(float(np.mean([r["correct"] for r in ctrl_res])))

    y_true = [s["correct_letter"] for s in mcq_samples]
    b_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in baseline_res]
    v_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in vir_res]
    b_probs = np.stack([r["probs"] for r in baseline_res])
    v_probs = np.stack([r["probs"] for r in vir_res])
    b_correct = [r["correct"] for r in baseline_res]
    v_correct = [r["correct"] for r in vir_res]

    baseline_acc = float(np.mean(b_correct))
    vir_acc = float(np.mean(v_correct))
    control_acc_mean = float(np.mean(control_accs))
    control_acc_std = float(np.std(control_accs))
    n_ge = sum(1 for a in control_accs if a >= vir_acc)
    perm_p = max((n_ge + 1) / (R_CTRL + 1), permutation_pvalue_floor(R_CTRL))
    p_vir_vs_baseline = mcnemar_p(b_correct, v_correct)

    stats = score_stats(raw_scores)
    summary = {
        "family": "internvl35", "checkpoint": tag, "model_id": model_id_str,
        "n_layers": n_layers, "n_heads": n_heads, "K": K,
        "topk_overlap_raw_vs_excess": overlap_frac,
        "vh_raw_mean": stats["mean"], "vh_raw_max": stats["max"],
        "baseline_acc": baseline_acc, "vir_acc": vir_acc,
        "control_acc_mean": control_acc_mean, "control_acc_std": control_acc_std,
        "p_vir_vs_baseline_mcnemar": p_vir_vs_baseline,
        "p_vir_vs_layer_matched_control": perm_p,
        "baseline_f1": macro_f1(y_true, b_pred), "vir_f1": macro_f1(y_true, v_pred),
        "baseline_auc": macro_auc(y_true, b_probs, OPTION_LETTERS), "vir_auc": macro_auc(y_true, v_probs, OPTION_LETTERS),
    }
    print(f"  [{tag}] baseline_acc={baseline_acc:.3f}  VIR_acc={vir_acc:.3f}  "
          f"control_acc={control_acc_mean:.3f}+-{control_acc_std:.3f}  "
          f"p(VIR vs control)={perm_p:.2e}  p(VIR vs baseline)={p_vir_vs_baseline:.2e}")

    free_gpu(raw_model, tokenizer)
    return summary

## Section 3 -- Pilot run (small N, catches bugs fast) -- run in internvl_env

In [3]:
PILOT_N_DISCOVERY = 20
PILOT_N_CAUSAL = 20
PILOT_R_CTRL_SAVE = R_CTRL
R_CTRL = 10

for tag, cfg in INTERNVL_CHECKPOINTS.items():
    result = run_internvl_checkpoint(tag, cfg, PILOT_N_DISCOVERY, PILOT_N_CAUSAL)
    all_results[("internvl35", tag + "_PILOT")] = result

R_CTRL = PILOT_R_CTRL_SAVE
pd.DataFrame([v for k, v in all_results.items() if k[0] == "internvl35" and "PILOT" in k[1]])


=== [internvl35_4b] Loading OpenGVLab/InternVL3_5-4B ===


`torch_dtype` is deprecated! Use `dtype` instead!


FlashAttention2 is not installed.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:00<00:00,  4.14it/s]

...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[internvl35_8b] layer-matched controls: 100%|██████████| 10/10 [00:47<00:00,  4.69s/it]

  [internvl35_8b] baseline_acc=0.250  VIR_acc=0.300  control_acc=0.355+-0.193  p(VIR vs control)=7.27e-01  p(VIR vs baseline)=1.00e+00


,family,checkpoint,model_id,n_layers,n_heads,K,topk_overlap_raw_vs_excess,vh_raw_mean,vh_raw_max,baseline_acc,vir_acc,control_acc_mean,control_acc_std,p_vir_vs_baseline_mcnemar,p_vir_vs_layer_matched_control,baseline_f1,vir_f1,baseline_auc,vir_auc
0,internvl35,internvl35_4b,OpenGVLab/InternVL3_5-4B,36,32,58,0.655172,0.032409,0.517560,0.25,0.35,0.260,0.213073,0.617075,0.272727,0.212500,0.272222,0.595242,0.525357
1,internvl35,internvl35_8b,OpenGVLab/InternVL3_5-8B,36,32,58,0.448276,0.031611,0.312252,0.25,0.30,0.355,0.192938,1.000000,0.727273,0.178788,0.200855,0.534849,0.570539


## Section 3 -- Main run (N=INTERNVL_N_DISCOVERY discovery, INTERNVL_N_CAUSAL causal, default 600/600) -- run in internvl_env

In [3]:
for tag, cfg in INTERNVL_CHECKPOINTS.items():
    try:
        result = run_internvl_checkpoint(tag, cfg, INTERNVL_N_DISCOVERY, INTERNVL_N_CAUSAL)
        all_results[("internvl35", tag)] = result
    except ValueError as e:
        print(f"[{tag}] SKIPPED -- layer-matched control infeasible at this K_FRACTION: {e}")
        free_gpu()
        continue

internvl_df = pd.DataFrame([v for k, v in all_results.items() if k[0] == "internvl35" and "PILOT" not in k[1]])
internvl_df.to_csv("logs/multistage_internvl35_results.csv", index=False)
print(internvl_df.to_string(index=False))


=== [internvl35_4b] Loading OpenGVLab/InternVL3_5-4B ===


`torch_dtype` is deprecated! Use `dtype` instead!


FlashAttention2 is not installed.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 21.31it/s]

...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[internvl35_8b] layer-matched controls: 100%|██████████| 5/5 [05:58<00:00, 72.40s/it]

  [internvl35_8b] baseline_acc=0.237  VIR_acc=0.333  control_acc=0.219+-0.018  p(VIR vs control)=1.67e-01  p(VIR vs baseline)=1.96e-05
    family    checkpoint                 model_id  n_layers  n_heads  K  topk_overlap_raw_vs_excess  vh_raw_mean  vh_raw_max  baseline_acc  vir_acc  control_acc_mean  control_acc_std  p_vir_vs_baseline_mcnemar  p_vir_vs_layer_matched_control  baseline_f1   vir_f1  baseline_auc  vir_auc
internvl35 internvl35_4b OpenGVLab/InternVL3_5-4B        36       32 58                    0.603448     0.031621    0.485227      0.206667 0.183333          0.272667         0.072277                   0.360196                        1.000000     0.146475 0.158530      0.461032 0.440668
internvl35 internvl35_8b OpenGVLab/InternVL3_5-8B        36       32 58                    0.344828     0.030240    0.331713      0.236667 0.333333          0.218667         0.018330                   0.000020                        0.166667     0.220284 0.327683      0.470112 0.584200


---
# Final summary: all families, all checkpoints

Note: since Section 3 (InternVL3.5) runs in a separate kernel/process
(`internvl_env`), `all_results` here only aggregates whichever sections were
executed in THIS kernel session. Cross-environment aggregation is done via
the per-section CSVs (`logs/multistage_{qwen25vl,gemma4,internvl35}_results.csv`),
not this in-memory dict, when Sections 1-2 and Section 3 are run as separate
processes.

In [ ]:
import glob

csv_paths = sorted(glob.glob("logs/multistage_*_results.csv"))
frames = [pd.read_csv(p) for p in csv_paths]
if frames:
    final_df = pd.concat(frames, ignore_index=True)
    pd.set_option("display.width", 200)
    print(final_df.to_string(index=False))
else:
    print("No result CSVs found yet -- run the sections above first.")